In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\tmp\\course-restructure-20261118\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0

# Разбор A. Истрадефиллин: pH, форма и измеримость растворимости

Вместо условного pKa используем Table 2 [Wang et al. (2022)](https://doi.org/10.3390/cryst12070917), CC BY 4.0: восемь значений при 293.15 K. Первичный PDF и supporting material доступны локально. Таблица содержит **µg/mL**, а не mg/mL. Репликатные ошибки для этих значений не опубликованы в таблице; весовую регрессию с вымышленными SD не проводим.

Вопрос: позволяют ли эти данные количественно оценить pKa или intrinsic S0? Сначала проверяем единицы, аналитическую область и solid-phase assumptions. Формула Henderson–Hasselbalch для равновесной растворимости слабого основания S=S0(1+10^(pKa−pH)) требует одной постоянной нейтральной твёрдой фазы, равновесия и применимого activity model.

In [2]:
from pathlib import Path
import sys,json,csv,math
import numpy as np
from scipy import stats
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUT=workspace('M04-L01','case');S_DATA=DATA/'istradefylline_solubility'
def read_csv(name):
    with (S_DATA/name).open(encoding='utf-8',newline='') as f:return list(csv.DictReader(f))
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=150,bbox_inches='tight');display(fig);plt.close(fig)


In [3]:
rows=read_csv('istradefylline_aqueous_pH_table2.csv')
pH=np.array([float(r['pH']) for r in rows]);S=np.array([float(r['solubility_ug_mL_as_reported']) for r in rows])
# 1 ug/mL = 1 mg/L = 0.001 g/L; molarity uses API-equivalent molecular mass.
MW=384.43;mass_g_L=S*.001;molar=mass_g_L/MW
display([{'pH':float(p),'reported_ug_mL':float(s),'API_equivalent_mol_L':float(c)} for p,s,c in zip(pH,S,molar)])
calibration_min_mg_mL=.001  # Stated calibration lower bound in the primary article.
below=S*.001<calibration_min_mg_mL
print('Below stated calibration without a documented concentration step:',below.tolist())


[{'pH': 1.0,
  'reported_ug_mL': 0.41,
  'API_equivalent_mol_L': 1.0665140597768124e-06},
 {'pH': 2.0,
  'reported_ug_mL': 0.39,
  'API_equivalent_mol_L': 1.0144889836901387e-06},
 {'pH': 3.0,
  'reported_ug_mL': 0.32,
  'API_equivalent_mol_L': 8.324012173867805e-07},
 {'pH': 4.0,
  'reported_ug_mL': 0.31,
  'API_equivalent_mol_L': 8.063886793434435e-07},
 {'pH': 7.0,
  'reported_ug_mL': 0.27,
  'API_equivalent_mol_L': 7.02338527170096e-07},
 {'pH': 8.0,
  'reported_ug_mL': 0.18,
  'API_equivalent_mol_L': 4.6822568478006395e-07},
 {'pH': 10.0,
  'reported_ug_mL': 0.11,
  'API_equivalent_mol_L': 2.8613791847670576e-07},
 {'pH': 12.0,
  'reported_ug_mL': 0.1,
  'API_equivalent_mol_L': 2.601253804333689e-07}]

Below stated calibration without a documented concentration step: [True, True, True, True, True, True, True, True]


Первичный текст сообщает диапазон HPLC-калибровки 0.001–0.1 mg/mL, то есть нижняя граница 1 µg/mL. Все восемь aqueous-значений ниже неё. Это запрос на проверку отдельной калибровки/LOQ или концентрирования, а не утверждение, что авторы обязательно измеряли неправильно. Соответствующие raw analytical details в нашем наборе отсутствуют.

Проверим **условное** следствие одной HH-модели: при выбранном постоянном S0 выражение pH+log10(S/S0−1) должно дать один pKa. Значение 0.10 µg/mL из самой таблицы используем только как диагностическое допущение, не как доказанный intrinsic S0. Там, где S≤S0, логарифм не определён; не заменяем это искусственным epsilon.

In [4]:
assumed_S0=.10
valid=S>assumed_S0
implied=np.full(len(S),np.nan)
implied[valid]=pH[valid]+np.log10(S[valid]/assumed_S0-1)
display([{'pH':float(p),'conditional_implied_pKa':None if not ok else float(v)} for p,ok,v in zip(pH,valid,implied)])
fig,axes=plt.subplots(1,2,figsize=(9,3.5))
axes[0].plot(pH,S,'o-');axes[0].set(xlabel='pH',ylabel='Published solubility / ug mL^-1')
axes[1].plot(pH[valid],implied[valid],'o');axes[1].set(xlabel='pH',ylabel='Implied pKa under unverified constant S0')
publish_plot(fig,'pH_model_diagnostic.png')
result={'kind':'calculated','scope':'published pH-solubility audit','API_equivalent_molarity':molar.tolist(),
        'conditional_HH_pKa_values':[float(v) for v in implied[valid]],'pKa_accepted':False,
        'intrinsic_S0_accepted':False,'analytical_domain':'REVIEW_REQUIRED','new_measurements':'NOT_RUN'}


[{'pH': 1.0, 'conditional_implied_pKa': 1.4913616938342726},
 {'pH': 2.0, 'conditional_implied_pKa': 2.462397997898956},
 {'pH': 3.0, 'conditional_implied_pKa': 3.342422680822206},
 {'pH': 4.0, 'conditional_implied_pKa': 4.3222192947339195},
 {'pH': 7.0, 'conditional_implied_pKa': 7.230448921378274},
 {'pH': 8.0, 'conditional_implied_pKa': 7.903089986991944},
 {'pH': 10.0, 'conditional_implied_pKa': 9.0},
 {'pH': 12.0, 'conditional_implied_pKa': None}]

<Figure size 900x350 with 2 Axes>

Условные implied pKa не совпадают. Одной этой диагностикой нельзя установить причину: возможны неадекватная модель, изменение фазы/состава, аналитические ограничения или неполное описание подготовки. Не выдаём диапазон значений за измеренную pKa. Для B аналогично различаем baseline при заданном pH, химическую форму и intrinsic neutral-species S0.

In [5]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
print(append_record(init_dossier(),'M04-L01','A',result,ROOT,inputs=['practicuum/shared/data/istradefylline_solubility/istradefylline_solubility_table_S1.csv', 'practicuum/shared/data/istradefylline_solubility/istradefylline_aqueous_pH_table2.csv', 'practicuum/shared/data/istradefylline_solubility/supporting-information.pdf', 'practicuum/shared/data/istradefylline_solubility/primary-article.pdf', 'practicuum/shared/data/istradefylline_solubility/result.json', 'practicuum/shared/data/source-responses/statin_stability_verified.xml']))


C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261119-professional-solubility-windows\workspace\my_research\evidence\M04-L01\20261119T075920116863Z-A-e18a9e9be22b4331a75fad7142a7340c.json
